# Query Modes - Pick the right query mode

Compare **global**, **local**, **naive**, and **entity_grounded** query modes
with timing benchmarks and LLM token/cost analysis.

**Run:**
```bash
dotenvx run -- uv run jupyter nbconvert --to notebook --execute --inplace notebooks/02_query_modes.ipynb
```


In [1]:
import json
import logging
import sqlite3
import time
from collections import defaultdict
from pathlib import Path

logging.basicConfig(level=logging.WARNING)
logging.getLogger("nano-graphrag").setLevel(logging.WARNING)

WORKING_DIR = Path("./_cache/02_query_modes")
WORKING_DIR.mkdir(parents=True, exist_ok=True)


## Imports and helpers

In [2]:
from nano_graphrag import GraphRAG, QueryParam
from nano_graphrag.base import ResponseType

def _get_cache_keys(db_path):
    if not db_path.exists():
        return set(), 0.0, 0, 0, 0
    conn = sqlite3.connect(str(db_path))
    rows = conn.execute("SELECT key, value FROM kv_store").fetchall()
    conn.close()
    keys = set()
    total_cost = 0.0
    total_prompt = 0
    total_completion = 0
    total_tokens = 0
    for key, val_str in rows:
        keys.add(key)
        data = json.loads(val_str)
        total_cost += float(data.get("cost_usd", 0))
        total_prompt += int(data.get("prompt_tokens", 0))
        total_completion += int(data.get("completion_tokens", 0))
        total_tokens += int(data.get("total_tokens", 0))
    return keys, total_cost, total_prompt, total_completion, total_tokens

def snapshot_cache(db_path):
    return _get_cache_keys(db_path)

def diff_cache(db_path, before):
    after = _get_cache_keys(db_path)
    new_keys = after[0] - before[0]
    return {
        "llm_calls": len(new_keys),
        "prompt_tokens": after[2] - before[2],
        "completion_tokens": after[3] - before[3],
        "total_tokens": after[4] - before[4],
        "cost_usd": after[1] - before[1],
    }


## Build index

In [3]:
rag = GraphRAG(working_dir=str(WORKING_DIR), enable_llm_cache=True, enable_naive_rag=True)

with open("../tests/fixtures/mock_data.txt", encoding="utf-8-sig") as f:
    text = f.read()[:6000]

print("Building index...")
await rag.ainsert(text)
print("Done.")


2026-05-17T11:51:24.225160Z [info     ] tokenizer_loading              [nano-graphrag] model_name=gpt-4o tokenizer_type=tiktoken


INFO:nano-graphrag:{'tokenizer_type': 'tiktoken', 'model_name': 'gpt-4o', 'event': 'tokenizer_loading', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T11:51:24.225160Z'}


2026-05-17T11:51:24.387644Z [info     ] sqlite_kv_loaded               [nano-graphrag] entries=0 namespace=llm_response_cache


INFO:nano-graphrag:{'namespace': 'llm_response_cache', 'entries': 0, 'event': 'sqlite_kv_loaded', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T11:51:24.387644Z'}


2026-05-17T11:51:24.388184Z [info     ] litellm_configured             [nano-graphrag] api_base=None model=openrouter/google/gemma-4-31b-it structured_output=True


...

2026-05-17T11:55:11.523293Z [info     ] community_report_progress      [nano-graphrag] processed=10 run_id=6c82e7e2


INFO:nano-graphrag:{'processed': 10, 'event': 'community_report_progress', 'run_id': '6c82e7e2', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T11:55:11.523293Z'}


2026-05-17T11:55:11.598478Z [info     ] graph_write                    [nano-graphrag] edges=13 nodes=21 run_id=6c82e7e2


INFO:nano-graphrag:{'nodes': 21, 'edges': 13, 'event': 'graph_write', 'run_id': '6c82e7e2', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T11:55:11.598478Z'}



Provider List: https://docs.litellm.ai/docs/providers

Done.


## Benchmark all four modes

In [4]:
questions = [
    "What are the main themes in A Christmas Carol?",
    "What is the relationship between Scrooge and Bob Cratchit?",
    "Describe the character of the Ghost of Christmas Yet to Come.",
    "How does Scrooge change throughout the story?",
    "What role does Tiny Tim play in the narrative?",
]
modes = ["global", "local", "naive", "entity_grounded"]

cache_db = WORKING_DIR / "kv_store_llm_response_cache.db"

print(f"Benchmarking {len(questions)} questions x {len(modes)} modes...")
print("=" * 80)
for q in questions:
    print(f"\nQ: {q[:80]}...")
    for mode in modes:
        before = snapshot_cache(cache_db) if cache_db.exists() else (set(), 0, 0, 0, 0)
        t0 = time.time()
        result = await rag.aquery(q, param=QueryParam(mode=mode))
        elapsed = time.time() - t0
        usage = diff_cache(cache_db, before)
        print(
            f"  [{mode:18s}] {elapsed:.1f}s | "
            f"{usage['llm_calls']} LLM calls | "
            f"{usage['total_tokens']} tokens | "
            f"${usage['cost_usd']:.4f}"
        )


2026-05-17T11:55:11.620510Z [info     ] query_start                    [nano-graphrag] mode=global query='What are the main themes in A Christmas Carol?' run_id=8dc8d27b


INFO:nano-graphrag:{'query': 'What are the main themes in A Christmas Carol?', 'event': 'query_start', 'run_id': '8dc8d27b', 'mode': 'global', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T11:55:11.620510Z'}


2026-05-17T11:55:11.623567Z [info     ] global_retrieved_communities   [nano-graphrag] count=10 mode=global run_id=8dc8d27b


INFO:nano-graphrag:{'count': 10, 'event': 'global_retrieved_communities', 'run_id': '8dc8d27b', 'mode': 'global', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T11:55:11.623567Z'}


2026-05-17T11:55:11.625525Z [info     ] global_search_groups           [nano-graphrag] groups=1 mode=global run_id=8dc8d27b


...

2026-05-17T11:56:49.362571Z [info     ] entity_grounded_query_complete [nano-graphrag] confidence=0.0 entities_retrieved=20 entities_used=0 mode=entity_grounded run_id=b917d9ac validation_errors=1


INFO:nano-graphrag:{'entities_retrieved': 20, 'entities_used': 0, 'confidence': 0.0, 'validation_errors': 1, 'event': 'entity_grounded_query_complete', 'run_id': 'b917d9ac', 'mode': 'entity_grounded', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T11:56:49.362571Z'}


2026-05-17T11:56:49.363148Z [info     ] query_complete                 [nano-graphrag] answer_chars=56 latency_ms=4128.3 mode=entity_grounded run_id=b917d9ac


INFO:nano-graphrag:{'latency_ms': 4128.3, 'answer_chars': 56, 'event': 'query_complete', 'run_id': 'b917d9ac', 'mode': 'entity_grounded', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T11:56:49.363148Z'}



Provider List: https://docs.litellm.ai/docs/providers

  [entity_grounded   ] 4.1s | 1 LLM calls | 420 tokens | $0.0000


## Summary

In [5]:

print("Mode comparison:")
print("  global           - best for broad, theme-level questions (uses community reports)")
print("  local            - best for specific entity/relationship questions (uses vector search)")
print("  naive            - simplest: embed query, retrieve chunks, answer directly")
print("  entity_grounded  - like local but validates entity matches before answering")
print()
print("Tips:")
print("  - Use `global` for broad theme questions")
print("  - Use `local` for entity-specific questions")
print("  - Use `naive` as a fast baseline or fallback")
print("  - Use `entity_grounded` for answer quality over recall")


Mode comparison:
  global           - best for broad, theme-level questions (uses community reports)
  local            - best for specific entity/relationship questions (uses vector search)
  naive            - simplest: embed query, retrieve chunks, answer directly
  entity_grounded  - like local but validates entity matches before answering

Tips:
  - Use `global` for broad theme questions
  - Use `local` for entity-specific questions
  - Use `naive` as a fast baseline or fallback
  - Use `entity_grounded` for answer quality over recall
